In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("../data/health.db")
df = con.execute("SELECT * FROM features").df()

print(f"Shape: {df.shape}")
df.head()

Shape: (1000, 23)


,patient_id,age,gender,insurance_encoded,gender_encoded,num_admissions_12m,total_length_of_stay,max_single_stay,avg_length_of_stay,num_chronic_conditions,...,has_hypertension,has_copd,num_medications,num_active_medications,avg_glucose,avg_systolic_bp,avg_creatinine,avg_hba1c,avg_heart_rate,readmitted_30days
0,4989a96a-dc04-49e4-9b9b-baf21ed26eb4,75,Male,0,0,2.0,8.0,4.0,4.0,0.0,...,0,0,2,2.0,334.420,166.770,3.340,5.94,60.79,0
1,b8d0b0ba-0766-4846-b42e-063ef0af83b8,59,Male,2,0,0.0,0.0,0.0,0.0,0.0,...,0,0,3,1.0,228.055,143.475,5.760,8.28,87.66,0
2,1939f749-398f-4d1d-b969-c53cc8649cfa,90,Female,1,1,0.0,0.0,0.0,0.0,1.0,...,0,0,2,1.0,304.090,143.160,5.660,8.28,122.62,0
3,a39fb550-3f9d-4038-b989-402f30f9cf6d,44,Male,2,0,2.0,23.0,13.0,11.5,1.0,...,0,0,3,0.0,164.550,122.830,3.430,11.66,87.66,0
4,73030d01-3138-4009-a944-045cb826dd55,84,Female,3,1,0.0,0.0,0.0,0.0,0.0,...,0,0,3,2.0,228.055,143.475,4.265,8.28,87.66,0


In [2]:
missing = df.isnull().sum()
missing[missing > 0]

Series([], dtype: int64)

In [3]:
df.describe().round(2)

,age,insurance_encoded,gender_encoded,num_admissions_12m,total_length_of_stay,max_single_stay,avg_length_of_stay,num_chronic_conditions,has_diabetes,has_heart_failure,...,has_hypertension,has_copd,num_medications,num_active_medications,avg_glucose,avg_systolic_bp,avg_creatinine,avg_hba1c,avg_heart_rate,readmitted_30days
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,...,1000.0,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,53.41,1.48,0.49,1.50,11.38,7.32,6.10,1.24,0.18,0.17,...,0.2,0.18,1.96,0.96,229.55,143.83,4.23,8.32,87.65,0.25
std,20.83,1.14,0.50,1.15,9.90,4.95,4.25,0.91,0.38,0.38,...,0.4,0.38,0.83,0.79,70.79,23.31,1.63,1.77,18.13,0.43
min,18.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.0,0.00,1.00,0.00,60.62,90.02,0.51,4.01,45.00,0.00
25%,35.00,0.00,0.00,1.00,3.00,3.00,2.50,1.00,0.00,0.00,...,0.0,0.00,1.00,0.00,195.78,133.23,3.50,7.48,78.96,0.00
50%,53.00,1.00,0.00,1.00,10.00,8.00,6.50,1.00,0.00,0.00,...,0.0,0.00,2.00,1.00,228.06,143.48,4.26,8.28,87.66,0.00
75%,71.00,3.00,1.00,2.00,17.00,12.00,9.50,2.00,0.00,0.00,...,0.0,0.00,3.00,1.00,259.05,154.39,4.91,9.09,96.82,0.00
max,91.00,3.00,1.00,5.00,49.00,14.00,14.00,4.00,1.00,1.00,...,1.0,1.00,3.00,3.00,397.99,199.96,7.97,12.99,129.91,1.00


In [4]:
label_counts = df["readmitted_30days"].value_counts()
print(label_counts)
print(f"\nReadmission rate: {label_counts[1] / len(df) * 100:.1f}%")

readmitted_30days
0    751
1    249
Name: count, dtype: int64

Readmission rate: 24.9%


In [5]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
correlations = (
    df[numeric_cols]
    .corr()["readmitted_30days"]
    .drop("readmitted_30days")
    .sort_values(ascending=False)
)
print(correlations.round(3))

num_admissions_12m        0.295
total_length_of_stay      0.245
max_single_stay           0.119
avg_hba1c                 0.069
avg_length_of_stay        0.049
avg_glucose               0.045
has_diabetes              0.039
gender_encoded            0.036
avg_creatinine            0.006
num_active_medications    0.003
has_ckd                   0.003
has_copd                 -0.000
num_chronic_conditions   -0.001
has_hypertension         -0.009
insurance_encoded        -0.010
age                      -0.016
avg_systolic_bp          -0.026
num_medications          -0.027
has_heart_failure        -0.045
avg_heart_rate           -0.053
Name: readmitted_30days, dtype: float64
